# LeetCode 1786 — Number of Restricted Paths From First to Last Node bằng Python

Link đề: https://leetcode.com/problems/number-of-restricted-paths-from-first-to-last-node/ · Độ khó: **Medium** · Chủ đề: Dynamic Programming, Graph, Topological Sort, Heap (Priority Queue), Shortest Path

Skeleton LeetCode:

```python
class Solution:
    def countRestrictedPaths(self, n: int, edges: List[List[int]]) -> int:
        ...
```

## 1. Mục tiêu học

1. Tính khoảng cách ngắn nhất từ **một đỉnh đích** tới mọi đỉnh bằng **Dijkstra** với `heapq`.
2. Nhận ra điều kiện "khoảng cách **giảm nghiêm ngặt** dọc đường đi" biến đồ thị vô hướng thành một **DAG** (đồ thị có hướng không chu trình).
3. Đếm đường đi trên DAG bằng **quy hoạch động**: đệ quy có ghi nhớ, hoặc duyệt đỉnh theo thứ tự khoảng cách tăng dần (không đệ quy).
4. Nắm các bẫy: điều kiện **nghiêm ngặt** (`>`), modulo `10⁹ + 7`, đệ quy sâu với `n = 2·10⁴`.
5. Đối chiếu với vét cạn liệt kê đường trên đồ thị ngẫu nhiên có seed cố định.

## 2. Đề bài & ví dụ

Cho đồ thị vô hướng, có trọng số, **liên thông** với `n` đỉnh đánh số `1..n`, và `edges[i] = [u, v, w]`. Một **đường đi** là dãy đỉnh `z_0, z_1, …, z_k` có cạnh giữa mỗi cặp liên tiếp; độ dài = tổng trọng số.

Gọi `distanceToLastNode(x)` là độ dài đường đi **ngắn nhất** từ đỉnh `n` tới `x`. Một đường đi là **restricted** nếu `distanceToLastNode(z_i) > distanceToLastNode(z_{i+1})` với mọi `i`.

Trả về số đường đi restricted từ đỉnh `1` tới đỉnh `n`, theo modulo `10⁹ + 7`.

**Ví dụ 1:** `n = 5, edges = [[1,2,3],[1,3,3],[2,3,1],[1,4,2],[5,2,2],[3,5,1],[5,4,10]]` → `3`: `1→2→5`, `1→2→3→5`, `1→3→5`.

**Ví dụ 2:** `n = 7, edges = [[1,3,1],[4,1,2],[7,3,4],[2,5,3],[5,6,1],[6,7,2],[7,5,3],[2,6,4]]` → `1`: chỉ `1→3→7`.

**Ràng buộc:** `1 ≤ n ≤ 2·10⁴`; `n − 1 ≤ edges.length ≤ 4·10⁴`; `1 ≤ w ≤ 10⁵`; không cạnh lặp, không khuyên; đồ thị liên thông.

In [1]:
import random
import time
import heapq
import sys
from collections import defaultdict
from functools import lru_cache
from typing import List

MOD = 10 ** 9 + 7
examples = [
    (5, [[1, 2, 3], [1, 3, 3], [2, 3, 1], [1, 4, 2], [5, 2, 2], [3, 5, 1], [5, 4, 10]], 3),
    (7, [[1, 3, 1], [4, 1, 2], [7, 3, 4], [2, 5, 3], [5, 6, 1], [6, 7, 2], [7, 5, 3], [2, 6, 4]], 1),
]
for n, e, exp in examples:
    print(f"n={n}, {len(e)} cạnh -> kỳ vọng {exp}")

n=5, 7 cạnh -> kỳ vọng 3
n=7, 8 cạnh -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 Danh sách kề có trọng số

In [2]:
def build_graph(n, edges):
    g = [[] for _ in range(n + 1)]
    for u, v, w in edges:
        g[u].append((v, w)); g[v].append((u, w))
    return g

g = build_graph(*examples[0][:2])
print({u: g[u] for u in range(1, 6)})

{1: [(2, 3), (3, 3), (4, 2)], 2: [(1, 3), (3, 1), (5, 2)], 3: [(1, 3), (2, 1), (5, 1)], 4: [(1, 2), (5, 10)], 5: [(2, 2), (3, 1), (4, 10)]}


### 3.2 Dijkstra với `heapq`

Từ nguồn `s`, `dist[s] = 0`; lặp: lấy đỉnh có `dist` nhỏ nhất chưa chốt, nới lỏng các cạnh. Bản "lazy": đẩy `(dist, đỉnh)` vào heap, khi lấy ra mà `d > dist[u]` thì bỏ qua (bản cũ). Độ phức tạp `O((V + E) log V)`.

In [3]:
def dijkstra(n, g, src):
    INF = float("inf")
    dist = [INF] * (n + 1); dist[src] = 0
    pq = [(0, src)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]:
            continue
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return dist

print("distanceToLastNode (Ví dụ 1):", dijkstra(5, g, 5)[1:])

distanceToLastNode (Ví dụ 1): [4, 2, 1, 6, 0]


### 3.3 DAG và đếm đường đi

Nếu chỉ đi theo cạnh `u → v` khi `dist[u] > dist[v]`, thì `dist` **giảm nghiêm ngặt** dọc mọi đường ⇒ không thể quay lại đỉnh cũ ⇒ **không có chu trình** ⇒ DAG. Trên DAG: `ways(u) = Σ ways(v)` qua các cạnh `u → v`, `ways(n) = 1`.

### 3.4 `functools.lru_cache` và giới hạn đệ quy

Đệ quy có ghi nhớ rất gọn nhưng độ sâu có thể tới `n = 2·10⁴` > 1000 (mặc định) ⇒ cần `sys.setrecursionlimit` hoặc cài bằng vòng lặp.

## 4. Ý tưởng nền tảng / trực giác

**Ví dụ 1:** khoảng cách tới đỉnh 5: `d = [_, 4, 2, 1, 6, 0]` (đỉnh 1..5).

Giữ lại các cạnh "đi xuống" (`d` giảm):

| cạnh | d hai đầu | hướng giữ lại |
|---|---|---|
| 1–2 (3) | 4, 2 | 1 → 2 |
| 1–3 (3) | 4, 1 | 1 → 3 |
| 2–3 (1) | 2, 1 | 2 → 3 |
| 1–4 (2) | 4, 6 | 4 → 1 |
| 5–2 (2) | 0, 2 | 2 → 5 |
| 3–5 (1) | 1, 0 | 3 → 5 |
| 5–4 (10) | 0, 6 | 4 → 5 |

Đếm ngược từ đỉnh 5: `ways(5) = 1`, `ways(3) = 1`, `ways(2) = ways(3) + ways(5) = 2`, `ways(1) = ways(2) + ways(3) = 3` ✓.

In [4]:
n, e, _ = examples[0]
d = dijkstra(n, build_graph(n, e), n)
print([(u, v, f"{u}->{v}" if d[u] > d[v] else (f"{v}->{u}" if d[v] > d[u] else "bỏ (bằng nhau)")) for u, v, w in e])

[(1, 2, '1->2'), (1, 3, '1->3'), (2, 3, '2->3'), (1, 4, '4->1'), (5, 2, '2->5'), (3, 5, '3->5'), (5, 4, '4->5')]


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi đường restricted bằng DFS

**Ý tưởng:** tính `dist` (Dijkstra), rồi DFS từ đỉnh 1, chỉ đi sang hàng xóm có `dist` nhỏ hơn; mỗi lần chạm đỉnh `n` đếm 1. **Không** ghi nhớ ⇒ liệt kê từng đường.

In [5]:
def count_bruteforce(n, edges):
    g = build_graph(n, edges); dist = dijkstra(n, g, n)
    cnt = 0
    stack = [1]
    while stack:
        u = stack.pop()
        if u == n:
            cnt += 1; continue
        for v, _ in g[u]:
            if dist[u] > dist[v]:
                stack.append(v)
    return cnt % MOD

for n, e, exp in examples:
    got = count_bruteforce(n, e); print(got, "kỳ vọng", exp); assert got == exp

3 kỳ vọng 3
1 kỳ vọng 1


**Độ phức tạp:** số đường restricted có thể **mũ** theo `n` (ví dụ đồ thị "bậc thang" nhân đôi số đường mỗi tầng) ⇒ không dùng được cho `n` lớn.

**Nút thắt:** cùng một đỉnh `v` được đi qua bởi rất nhiều đường, ta lại đếm lại toàn bộ phần đường từ `v` tới `n` mỗi lần. Số đường từ `v` tới `n` là **cố định** ⇒ ghi nhớ!

In [6]:
def ladder(k):
    """Đồ thị bậc thang: mỗi tầng có 2 đỉnh, số đường restricted gấp đôi mỗi tầng (ví dụ tự dựng)."""
    n = 2 * k + 2; edges = []; s, t = 1, n
    prev = [s]
    for layer in range(k):
        a, b = 2 + 2 * layer, 3 + 2 * layer
        for p in prev:
            edges += [[p, a, 1], [p, b, 1]]
        prev = [a, b]
    for p in prev:
        edges.append([p, t, 1])
    return n, edges

for k in [4, 8, 12]:
    n, e = ladder(k); t0 = time.perf_counter(); r = count_bruteforce(n, e)
    print(f"bậc thang {k} tầng: {r} đường ({(time.perf_counter()-t0)*1000:.0f} ms)")

bậc thang 4 tầng: 16 đường (0 ms)
bậc thang 8 tầng: 256 đường (0 ms)
bậc thang 12 tầng: 4096 đường (2 ms)


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (DAG):** cạnh `u → v` hợp lệ khi `dist[u] > dist[v]`. Vì `dist` giảm nghiêm ngặt, đồ thị các cạnh hợp lệ là DAG; đường restricted = đường đi trong DAG.

**Insight 2 (DP đếm đường):** `ways(n) = 1`; `ways(u) = Σ_{v kề u, dist[v] < dist[u]} ways(v)`. Mỗi đỉnh tính **một lần**.

**Insight 3 (thứ tự topo miễn phí):** sắp đỉnh theo `dist` **tăng dần** là một thứ tự topo ngược của DAG: khi tính `ways(u)`, mọi `v` có `dist[v] < dist[u]` đã được tính. Hơn nữa, Dijkstra **đã** lấy đỉnh ra theo đúng thứ tự `dist` tăng dần ⇒ có thể làm DP ngay trong vòng Dijkstra.

**Quy tắc:** chỉ cộng `ways(v)` khi `dist[v] < dist[u]` **nghiêm ngặt**; đỉnh có `dist` bằng nhau không nối với nhau trong DAG.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Dijkstra + DFS có ghi nhớ

In [7]:
def count_memo(n, edges):
    g = build_graph(n, edges); dist = dijkstra(n, g, n)
    sys.setrecursionlimit(max(10000, 4 * n))
    @lru_cache(maxsize=None)
    def ways(u):
        if u == n:
            return 1
        return sum(ways(v) for v, _ in g[u] if dist[u] > dist[v]) % MOD
    res = ways(1)
    ways.cache_clear()
    return res

print([count_memo(n, e) for n, e, _ in examples])

[3, 1]


**Code nộp LeetCode (Cách 2):** đổi thành `countRestrictedPaths`. **Độ phức tạp:** Dijkstra `O((V+E) log V)` + DP `O(V + E)`. Rủi ro: độ sâu đệ quy tới `V`.

### 7.2 Cách 3 — Dijkstra + DP theo thứ tự `dist` tăng dần (không đệ quy)

In [8]:
def count_sorted_dp(n, edges):
    g = build_graph(n, edges); dist = dijkstra(n, g, n)
    order = sorted(range(1, n + 1), key=lambda u: dist[u])
    ways = [0] * (n + 1); ways[n] = 1
    for u in order:
        if u == n:
            continue
        ways[u] = sum(ways[v] for v, _ in g[u] if dist[v] < dist[u]) % MOD
        if u == 1:
            break                                  # không cần tính các đỉnh xa hơn đỉnh 1
    return ways[1]

print([count_sorted_dp(n, e) for n, e, _ in examples])

[3, 1]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** `O((V + E) log V)` (sắp xếp + Dijkstra), DP `O(V + E)`, Space `O(V + E)`.

### 7.3 Cách 4 — Gộp DP vào ngay vòng Dijkstra

Khi một đỉnh `u` được **chốt** (lấy ra khỏi heap với `d == dist[u]`), mọi đỉnh có `dist` nhỏ hơn đã được chốt trước ⇒ tính `ways[u]` ngay lúc đó.

In [9]:
def count_in_dijkstra(n, edges):
    g = build_graph(n, edges); INF = float("inf")
    dist = [INF] * (n + 1); dist[n] = 0
    ways = [0] * (n + 1); done = [False] * (n + 1)
    pq = [(0, n)]
    while pq:
        d, u = heapq.heappop(pq)
        if done[u]:
            continue
        done[u] = True
        ways[u] = 1 if u == n else sum(ways[v] for v, _ in g[u] if done[v] and dist[v] < d) % MOD
        if u == 1:
            return ways[1]
        for v, w in g[u]:
            if d + w < dist[v]:
                dist[v] = d + w; heapq.heappush(pq, (dist[v], v))
    return ways[1]

print([count_in_dijkstra(n, e) for n, e, _ in examples])

[3, 1]


Điều kiện `done[v] and dist[v] < d`: một hàng xóm `v` có `dist[v] < dist[u]` chắc chắn đã được chốt trước `u` (Dijkstra chốt theo `dist` không giảm). Kiểm `done[v]` để chắc rằng `dist[v]` là giá trị cuối cùng.

**Độ phức tạp Cách 4:** như Cách 3 nhưng có thể dừng sớm khi chốt đỉnh 1.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1**: thứ tự đỉnh theo `dist`, và `ways[u]` tính từ các hàng xóm "thấp hơn".

In [10]:
def trace_sorted_dp(n, edges):
    g = build_graph(n, edges); dist = dijkstra(n, g, n)
    order = sorted(range(1, n + 1), key=lambda u: dist[u])
    print("dist:", {u: dist[u] for u in range(1, n + 1)}, "| thứ tự:", order)
    ways = [0] * (n + 1); ways[n] = 1
    for u in order:
        if u == n:
            print(f"  ways[{n}] = 1 (đích)"); continue
        lower = [(v, ways[v]) for v, _ in g[u] if dist[v] < dist[u]]
        ways[u] = sum(x for _, x in lower) % MOD
        print(f"  ways[{u}] = tổng ways của hàng xóm thấp hơn {lower} = {ways[u]}")
    return ways[1]

assert trace_sorted_dp(*examples[0][:2]) == 3

dist: {1: 4, 2: 2, 3: 1, 4: 6, 5: 0} | thứ tự: [5, 3, 2, 1, 4]
  ways[5] = 1 (đích)
  ways[3] = tổng ways của hàng xóm thấp hơn [(5, 1)] = 1
  ways[2] = tổng ways của hàng xóm thấp hơn [(3, 1), (5, 1)] = 2
  ways[1] = tổng ways của hàng xóm thấp hơn [(2, 2), (3, 1)] = 3
  ways[4] = tổng ways của hàng xóm thấp hơn [(1, 3), (5, 1)] = 4


| đỉnh | dist | hàng xóm thấp hơn | ways |
|---|---|---|---|
| 5 | 0 | — | 1 |
| 3 | 1 | 5 | 1 |
| 2 | 2 | 3, 5 | 2 |
| 1 | 4 | 2, 3 | **3** |
| 4 | 6 | 1, 5 | 4 |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — Dijkstra + DP theo thứ tự khoảng cách (không đệ quy).

In [11]:
class Solution:
    def countRestrictedPaths(self, n: int, edges: List[List[int]]) -> int:
        MOD = 10 ** 9 + 7
        g = [[] for _ in range(n + 1)]
        for u, v, w in edges:
            g[u].append((v, w))
            g[v].append((u, w))
        INF = float("inf")
        dist = [INF] * (n + 1)
        dist[n] = 0
        pq = [(0, n)]
        order = []
        while pq:
            d, u = heapq.heappop(pq)
            if d > dist[u]:
                continue
            order.append(u)                        # thứ tự chốt = thứ tự dist tăng dần
            for v, w in g[u]:
                if d + w < dist[v]:
                    dist[v] = d + w
                    heapq.heappush(pq, (dist[v], v))
        ways = [0] * (n + 1)
        ways[n] = 1
        for u in order:
            if u != n:
                ways[u] = sum(ways[v] for v, _ in g[u] if dist[v] < dist[u]) % MOD
            if u == 1:
                break
        return ways[1]

for n, e, exp in examples:
    got = Solution().countRestrictedPaths(n, e); print(got, exp); assert got == exp

3 3
1 1


Lưu ý: với Dijkstra "lazy", một đỉnh có thể bị lấy ra nhiều lần với giá trị cũ; điều kiện `d > dist[u]` loại bản cũ, nhưng nếu có bản trùng với **đúng** `dist[u]` (đẩy hai lần cùng giá trị) thì `u` có thể vào `order` hai lần — vô hại vì `ways[u]` tính lại cho cùng kết quả.

## 10. Vì sao thuật toán đúng

**Bổ đề 1 (DAG).** Đặt `u → v` nếu `{u, v}` là cạnh và `dist[u] > dist[v]`. Một đường restricted là đúng một đường đi có hướng trong đồ thị này, và ngược lại. Vì `dist` giảm nghiêm ngặt dọc mọi đường có hướng, không có chu trình.

**Bổ đề 2 (công thức đếm).** Gọi `W(u)` = số đường restricted từ `u` tới `n`. `W(n) = 1` (đường một đỉnh). Với `u ≠ n`, mỗi đường bắt đầu bằng một cạnh `u → v` rồi là một đường restricted từ `v` — phân hoạch theo `v` ⇒ `W(u) = Σ_{u → v} W(v)`. (Từ `n` không có đường nào khác vì `dist[n] = 0` nhỏ nhất, không đi tiếp được.)

**Thứ tự tính.** Khi duyệt `u` theo `dist` tăng dần, mọi `v` với `dist[v] < dist[u]` đã được tính ⇒ `ways[u] = W(u)`. Modulo không ảnh hưởng tính đúng vì phép cộng tương thích với `mod`. ∎

**Dijkstra đúng** vì mọi trọng số dương.

## 11. Phân tích Time / Space Complexity

Gọi `V = n ≤ 2·10⁴`, `E = |edges| ≤ 4·10⁴`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê đường | `O((V+E) log V + số đường · độ dài)` — mũ | `O(V + E)` |
| 2. Dijkstra + memo DFS | `O((V+E) log V)` | `O(V + E)` + ngăn xếp `O(V)` |
| 3. Dijkstra + DP theo dist | `O((V+E) log V)` | `O(V + E)` |
| 4. DP trong Dijkstra | `O((V+E) log V)` | `O(V + E)` |

In [12]:
def random_connected(n, extra, rng, wmax=10 ** 5):
    edges, seen = [], set()
    for v in range(2, n + 1):
        u = rng.randint(1, v - 1); edges.append([u, v, rng.randint(1, wmax)]); seen.add((u, v))
    while len(edges) < n - 1 + extra:
        u, v = rng.sample(range(1, n + 1), 2)
        if (min(u, v), max(u, v)) not in seen:
            seen.add((min(u, v), max(u, v))); edges.append([u, v, rng.randint(1, wmax)])
    return edges

rng = random.Random(1786)
n = 20000; e = random_connected(n, 20000, rng, wmax=3)
for name, f in [("Cách 2 (memo)", count_memo), ("Cách 3 (sorted DP)", count_sorted_dp), ("Cách 4 (trong Dijkstra)", count_in_dijkstra), ("Bản nộp", Solution().countRestrictedPaths)]:
    t = time.perf_counter(); r = f(n, e); print(f"{name:<24} {r} ({(time.perf_counter()-t)*1000:.0f} ms)")

Cách 2 (memo)            1 (73 ms)


Cách 3 (sorted DP)       1 (50 ms)

Cách 4 (trong Dijkstra)  1 (16 ms)


Bản nộp                  1 (89 ms)


## 12. Bộ test

In [13]:
ALL = {"memo": count_memo, "sorted": count_sorted_dp, "inDijkstra": count_in_dijkstra, "submit": Solution().countRestrictedPaths}
def run_all(n, edges, with_brute=True):
    res = {k: f(n, edges) for k, f in ALL.items()}
    if with_brute:
        res["brute"] = count_bruteforce(n, edges)
    assert len(set(res.values())) == 1, (n, edges, res)
    return next(iter(res.values()))

for n, e, exp in examples:
    assert run_all(n, e) == exp
edge = [(1, [], 1), (2, [[1, 2, 5]], 1), (3, [[1, 2, 1], [2, 3, 1], [1, 3, 2]], 2), (3, [[1, 2, 1], [2, 3, 1], [1, 3, 1]], 1)]
for n, e, exp in edge:
    assert run_all(n, e) == exp, (n, e)
for k in [3, 6, 10]:
    n, e = ladder(k); assert run_all(n, e) == 2 ** k
rng = random.Random(17860)
for _ in range(400):
    n = rng.randint(2, 12)
    e = random_connected(n, rng.randint(0, min(10, n * (n - 1) // 2 - (n - 1))), rng, wmax=rng.choice([1, 3, 10]))
    run_all(n, e)
print("Ví dụ, edge case, đồ thị bậc thang (2^k đường), 400 đồ thị ngẫu nhiên (nhiều dist bằng nhau): OK")

Ví dụ, edge case, đồ thị bậc thang (2^k đường), 400 đồ thị ngẫu nhiên (nhiều dist bằng nhau): OK


Edge case `n = 1`: đỉnh 1 cũng là đỉnh `n` ⇒ có đúng **một** đường (đường một đỉnh). Đề có `n − 1 ≤ edges.length` nên `n = 1` cho phép mảng cạnh rỗng. Trường hợp `(3, [[1,2,1],[2,3,1],[1,3,1]])`: `dist = [_, 1, 1, 0]` — đỉnh 1 và 2 cùng `dist` nên cạnh 1–2 **không** dùng được ⇒ chỉ đường `1→3`.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `≥` thay vì `>` (cho phép đi giữa hai đỉnh cùng khoảng cách)

In [14]:
def wrong_ge(n, edges):
    g = build_graph(n, edges); dist = dijkstra(n, g, n)
    cnt = 0; stack = [(1, frozenset([1]))]
    while stack:
        u, seen = stack.pop()
        if u == n:
            cnt += 1; continue
        for v, _ in g[u]:
            if dist[u] >= dist[v] and v not in seen:              # SAI: phải là '>'
                stack.append((v, seen | {v}))
    return cnt

n, e = 3, [[1, 2, 1], [2, 3, 1], [1, 3, 1]]       # ví dụ tự dựng (đối chiếu lại đề gốc)
print("sai :", wrong_ge(n, e), "| đúng:", Solution().countRestrictedPaths(n, e))

sai : 2 | đúng: 1


**Vì sao sai:** đề yêu cầu khoảng cách **giảm nghiêm ngặt**; `1 → 2` có `dist` bằng nhau nên không được đi. (Với `≥` còn có thể tạo chu trình — phải thêm `seen` mới không lặp vô hạn.) **Sửa:** `dist[u] > dist[v]`.

### Lỗi 2 — Tính khoảng cách từ đỉnh `1` thay vì từ đỉnh `n`

In [15]:
def wrong_source(n, edges):
    g = build_graph(n, edges); dist1 = dijkstra(n, g, 1)
    dist = [dist1[n] - x for x in dist1]                    # SAI: "khoảng cách tới n" ≠ dist[n] - dist từ 1
    order = sorted(range(1, n + 1), key=lambda u: dist[u])
    ways = [0] * (n + 1); ways[n] = 1
    for u in order:
        if u != n:
            ways[u] = sum(ways[v] for v, _ in g[u] if dist[v] < dist[u]) % MOD
    return ways[1]

n, e, exp = examples[0]
print("sai :", wrong_source(n, e), "| đúng:", exp)

sai : 3 | đúng: 3


**Vì sao sai:** `distanceToLastNode(x)` là khoảng cách ngắn nhất **từ `n`** tới `x` — không suy ra được từ khoảng cách từ đỉnh 1. **Sửa:** Dijkstra với nguồn `n`.

### Lỗi 3 — Dùng BFS (bỏ qua trọng số)

In [16]:
from collections import deque
def wrong_bfs(n, edges):
    g = build_graph(n, edges)
    dist = [float("inf")] * (n + 1); dist[n] = 0; q = deque([n])
    while q:
        u = q.popleft()
        for v, _ in g[u]:
            if dist[v] == float("inf"):
                dist[v] = dist[u] + 1; q.append(v)
    order = sorted(range(1, n + 1), key=lambda u: dist[u])
    ways = [0] * (n + 1); ways[n] = 1
    for u in order:
        if u != n:
            ways[u] = sum(ways[v] for v, _ in g[u] if dist[v] < dist[u]) % MOD
    return ways[1]

for n, e, exp in examples:
    print("sai :", wrong_bfs(n, e), "| đúng:", exp)

sai : 3 | đúng: 3
sai : 1 | đúng: 1


**Vì sao sai:** BFS đếm số **cạnh**, không phải tổng trọng số. **Sửa:** Dijkstra.

### Lỗi 4 — DP theo thứ tự đỉnh `1..n` (không theo khoảng cách)

In [17]:
def wrong_order(n, edges):
    g = build_graph(n, edges); dist = dijkstra(n, g, n)
    ways = [0] * (n + 1); ways[n] = 1
    for u in range(n - 1, 0, -1):                           # SAI: thứ tự chỉ số, không phải thứ tự topo
        ways[u] = sum(ways[v] for v, _ in g[u] if dist[v] < dist[u]) % MOD
    return ways[1]

n, e, exp = examples[0]
print("Ví dụ 1 — sai :", wrong_order(n, e), "| đúng:", exp, "(tình cờ trùng)")
n, e = 4, [[1, 3, 1], [3, 2, 1], [2, 4, 1]]       # ví dụ tự dựng (đối chiếu lại đề gốc): đường 1 -> 3 -> 2 -> 4
print("Ví dụ tự dựng — sai :", wrong_order(n, e), "| đúng:", Solution().countRestrictedPaths(n, e), "| brute:", count_bruteforce(n, e))

Ví dụ 1 — sai : 3 | đúng: 3 (tình cờ trùng)
Ví dụ tự dựng — sai : 0 | đúng: 1 | brute: 1


**Vì sao sai:** DP cần mọi hàng xóm "thấp hơn" đã được tính trước. Ở Ví dụ 1 thứ tự chỉ số `4, 3, 2, 1` tình cờ vẫn hợp lệ; nhưng với đường `1 → 3 → 2 → 4`, khi tính `ways[3]` thì `ways[2]` (đỉnh thấp hơn) chưa có ⇒ `ways[3] = 0` ⇒ đáp án 0 thay vì 1. **Sửa:** duyệt theo `dist` tăng dần (hoặc thứ tự chốt của Dijkstra).

### Lỗi 5 — Quên modulo khi số đường rất lớn

In [18]:
def wrong_no_mod(n, edges):
    g = build_graph(n, edges); dist = dijkstra(n, g, n)
    ways = [0] * (n + 1); ways[n] = 1
    for u in sorted(range(1, n + 1), key=lambda u: dist[u]):
        if u != n:
            ways[u] = sum(ways[v] for v, _ in g[u] if dist[v] < dist[u])      # SAI: thiếu % MOD
    return ways[1]

n, e = ladder(40)                                   # 2^40 đường restricted
print("sai (không mod):", wrong_no_mod(n, e), "| đúng:", Solution().countRestrictedPaths(n, e), "| 2^40 % MOD =", 2 ** 40 % MOD)

sai (không mod): 1099511627776 | đúng: 511620083 | 2^40 % MOD = 511620083


**Vì sao sai:** đề yêu cầu kết quả `mod 10⁹ + 7`; Python không tràn số nhưng trả số thô là **sai đáp án** (và trong C++/Java còn tràn). **Sửa:** `% MOD` sau mỗi tổng.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm đường đi mà một đại lượng **giảm nghiêm ngặt**" ⇒ đồ thị trở thành **DAG** ⇒ DP đếm đường.
- Đại lượng là khoảng cách ngắn nhất có trọng số ⇒ **Dijkstra** từ đúng nguồn.
- Thứ tự topo = sắp theo đại lượng đó (miễn phí từ Dijkstra).
- `n` lớn ⇒ tránh đệ quy sâu.
- Nhớ modulo.

## 15. Biến thể của bài

1. **1976 Number of Ways to Arrive at Destination**: đếm số **đường ngắn nhất** — DP trong Dijkstra với điều kiện `dist[u] + w == dist[v]`.
2. **Điều kiện tăng nghiêm ngặt** theo khoảng cách từ đỉnh 1: đối xứng.
3. **Đếm đường trên DAG bất kỳ** (797 All Paths From Source to Target là liệt kê).
4. **Tìm đường restricted dài nhất / ngắn nhất**: DP max/min thay cho tổng.
5. **Truy vấn nhiều đỉnh xuất phát**: `ways[u]` đã tính cho mọi `u` — trả lời `O(1)` mỗi truy vấn (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê | DFS mọi đường | mũ | `O(V+E)` | Kiểm chứng |
| 2. Memo DFS | Ghi nhớ `ways(u)` | `O((V+E) log V)` | `O(V+E)` + stack | Code ngắn, cần tăng recursionlimit |
| 3. DP theo dist | Sắp đỉnh theo dist | `O((V+E) log V)` | `O(V+E)` | **Bản nộp** |
| 4. DP trong Dijkstra | Tính khi chốt | `O((V+E) log V)` | `O(V+E)` | Dừng sớm khi chốt đỉnh 1 |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `n=2, [[1,2,7]]`; (b) `n=4, [[1,2,1],[2,4,1],[1,3,1],[3,4,1]]`; (c) `n=4, [[1,2,1],[2,3,1],[3,4,1]]`; (d) `n=3, [[1,2,5],[2,3,1],[1,3,1]]`.

**Bài 2 (tính tay):** Với **Ví dụ 2**, tính `distanceToLastNode` của mọi đỉnh và giải thích vì sao chỉ có một đường restricted.

**Bài 3 (điền TODO):** Hoàn thiện `downward_edges(n, edges)` trả list cạnh có hướng `(u, v)` với `dist[u] > dist[v]`.

**Bài 4 (tự viết lại):** Viết `ways_all(n, edges)` trả list `ways[u]` cho **mọi** đỉnh `u` (số đường restricted từ `u` tới `n`).

**Bài 5 (mini challenge):** Viết `count_shortest_paths(n, edges)` đếm số **đường ngắn nhất** từ 1 tới `n` (LeetCode 1976 với đồ thị đánh số 1..n), đối chiếu liệt kê.

In [19]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def downward_edges_todo(n, edges):
    dist = dijkstra(n, build_graph(n, edges), n)
    res = []
    for u, v, w in edges:
        # TODO: nếu dist[u] > dist[v] thêm (u, v); nếu dist[v] > dist[u] thêm (v, u)
        pass
    return res

In [20]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def ways_all(n, edges):
    # TODO: như Cách 3 nhưng không dừng ở đỉnh 1
    pass

def count_shortest_paths(n, edges):
    # TODO: Dijkstra từ 1; khi d + w == dist[v] thì cnt[v] += cnt[u]; khi nhỏ hơn thì cnt[v] = cnt[u]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: Dijkstra từ `n`. Mức 2: giữ cạnh đi xuống. Mức 3: (a) `1`; (b) `2`; (c) `1`; (d) `1` (`dist = [_, 1, 1, 0]`, cạnh 1–2 bị loại).

**Bài 2.** Mức 1: `dist` từ 7: `7:0, 3:4, 6:2, 5:3, 1:5, 2:6, 4:7`. Mức 2: từ 1 (dist 5) chỉ đi xuống được 3 (dist 4) — đỉnh 4 có dist 7. Mức 3: từ 3 chỉ đi xuống tới 7 ⇒ một đường `1→3→7`.

**Bài 3.** Mức 1: tính `dist`. Mức 2: so hai đầu cạnh. Mức 3: hai đầu bằng nhau ⇒ bỏ.

**Bài 4.** Mức 1: thứ tự theo `dist`. Mức 2: không `break`. Mức 3: kiểm `ways_all(...)[1] == countRestrictedPaths(...)`.

**Bài 5.** Mức 1: `cnt[1] = 1`. Mức 2: khi nới lỏng được `dist[v]` thì `cnt[v] = cnt[u]`; khi bằng thì cộng. Mức 3: chỉ xử lý mỗi đỉnh một lần (bỏ bản cũ trong heap).

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
b1 = [Solution().countRestrictedPaths(*a) for a in [(2, [[1, 2, 7]]), (4, [[1, 2, 1], [2, 4, 1], [1, 3, 1], [3, 4, 1]]),
                                                     (4, [[1, 2, 1], [2, 3, 1], [3, 4, 1]]), (3, [[1, 2, 5], [2, 3, 1], [1, 3, 1]])]]
print("Bài 1:", b1); assert b1 == [1, 2, 1, 1]
n, e, _ = examples[1]
d2 = dijkstra(n, build_graph(n, e), n)
print("Bài 2: dist =", {u: d2[u] for u in range(1, n + 1)}); assert d2[1:] == [5, 6, 4, 7, 3, 2, 0]
def downward_edges_todo(n, edges):
    dist = dijkstra(n, build_graph(n, edges), n); res = []
    for u, v, w in edges:
        if dist[u] > dist[v]: res.append((u, v))
        elif dist[v] > dist[u]: res.append((v, u))
    return res
print("Bài 3:", downward_edges_todo(*examples[0][:2]))

Bài 1: [1, 2, 1, 1]
Bài 2: dist = {1: 5, 2: 6, 3: 4, 4: 7, 5: 3, 6: 2, 7: 0}
Bài 3: [(1, 2), (1, 3), (2, 3), (4, 1), (2, 5), (3, 5), (4, 5)]


In [22]:
# Bài 4, 5
def ways_all(n, edges):
    g = build_graph(n, edges); dist = dijkstra(n, g, n)
    ways = [0] * (n + 1); ways[n] = 1
    for u in sorted(range(1, n + 1), key=lambda u: dist[u]):
        if u != n:
            ways[u] = sum(ways[v] for v, _ in g[u] if dist[v] < dist[u]) % MOD
    return ways

def count_shortest_paths(n, edges):
    g = build_graph(n, edges); INF = float("inf")
    dist = [INF] * (n + 1); cnt = [0] * (n + 1); dist[1] = 0; cnt[1] = 1
    pq = [(0, 1)]
    while pq:
        d, u = heapq.heappop(pq)
        if d > dist[u]: continue
        for v, w in g[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd; cnt[v] = cnt[u]; heapq.heappush(pq, (nd, v))
            elif nd == dist[v]:
                cnt[v] = (cnt[v] + cnt[u]) % MOD
    return cnt[n]

def shortest_brute(n, edges):
    g = build_graph(n, edges); best = [float("inf")]; c = [0]
    def dfs(u, seen, length):
        if length > best[0]: return
        if u == n:
            if length < best[0]: best[0], c[0] = length, 1
            else: c[0] += 1
            return
        for v, w in g[u]:
            if v not in seen:
                seen.add(v); dfs(v, seen, length + w); seen.remove(v)
    dfs(1, {1}, 0); return c[0]

rng = random.Random(9)
for _ in range(200):
    n = rng.randint(2, 8); e = random_connected(n, rng.randint(0, min(6, n * (n - 1) // 2 - (n - 1))), rng, wmax=3)
    wa = ways_all(n, e)
    assert wa[1] == Solution().countRestrictedPaths(n, e)
    assert count_shortest_paths(n, e) == shortest_brute(n, e), (n, e)
print("Bài 4, 5: OK | ways_all(Ví dụ 1) =", ways_all(*examples[0][:2])[1:], "| số đường ngắn nhất 1->5:", count_shortest_paths(*examples[0][:2]))

Bài 4, 5: OK | ways_all(Ví dụ 1) = [3, 2, 1, 4, 1] | số đường ngắn nhất 1->5: 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "Dijkstra từ đích + DP đếm đường trên DAG":**

```python
dist = dijkstra(from = n)            # ghi lại thứ tự chốt `order`
ways = [0]*(n+1); ways[n] = 1
for u in order:                      # dist tăng dần
    if u != n:
        ways[u] = sum(ways[v] for v, _ in g[u] if dist[v] < dist[u]) % MOD
    if u == 1: break
return ways[1]
```

**Checklist:**
- [ ] Dijkstra từ **đỉnh `n`** (không phải 1), dùng trọng số (không BFS)?
- [ ] Điều kiện **nghiêm ngặt** `dist[v] < dist[u]`?
- [ ] DP theo thứ tự `dist` tăng dần (không theo chỉ số)?
- [ ] Không đệ quy sâu với `n = 2·10⁴` (hoặc đã tăng `recursionlimit`)?
- [ ] `% (10**9 + 7)`?
- [ ] Đã test `n = 1`, các đỉnh cùng `dist`, đồ thị bậc thang (số đường mũ)?